In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_date') AS
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d AS calendar_date,
    date_format(d, 'EEEE') AS day_name,
    weekofyear(d) AS iso_week,
    dayofweek(d) IN (1, 7) AS is_weekend
FROM (
    SELECT explode(sequence(DATE '2026-04-01', DATE '2026-04-30', INTERVAL 1 DAY)) AS d
)

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_station') AS
SELECT * FROM IDENTIFIER(:catalog || '.silver.stations')

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_charger') AS
SELECT c.charger_id, c.station_id, c.connector_type, c.max_kw, c.is_active, c.commissioned_on, s.city
FROM IDENTIFIER(:catalog || '.silver.chargers') c
JOIN IDENTIFIER(:catalog || '.silver.stations') s ON c.station_id = s.station_id

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_member') AS
SELECT
    CAST(row_number() OVER (ORDER BY member_id) AS BIGINT) AS member_sk,
    member_id, full_name, email, phone, city, plan, joined_on,
    effective_from, effective_to, is_current
FROM IDENTIFIER(:catalog || '.silver.members');

ALTER TABLE IDENTIFIER(:catalog || '.gold.dim_member') ALTER COLUMN member_sk SET NOT NULL;

In [0]:
-- SELECT * FROM vp_dp20040222.gold.dim_date;

In [0]:
-- SELECT * FROM vp_dp20040222.gold.dim_charger;

In [0]:
-- SELECT * FROM vp_dp20040222.gold.dim_member;


In [0]:
-- SELECT * FROM vp_dp20040222.gold.dim_station;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.fact_sessions') AS
WITH session_base AS (
    SELECT
        s.session_id,
        s.session_date,
        d.date_key,
        c.charger_id,
        c.connector_type,
        st.station_id,
        s.member_id,
        s.status,
        s.time_band,
        s.energy_kwh,
        s.duration_minutes,
        s.idle_minutes,
        s.amount_paid
    FROM IDENTIFIER(:catalog || '.silver.sessions') s
    JOIN IDENTIFIER(:catalog || '.gold.dim_date') d
        ON s.session_date = d.calendar_date
    JOIN IDENTIFIER(:catalog || '.silver.chargers') c
        ON s.charger_id = c.charger_id
    JOIN IDENTIFIER(:catalog || '.silver.stations') st
        ON c.station_id = st.station_id
),
member_version AS (
    SELECT
        sb.*,
        dm.member_sk,
        COALESCE(dm.plan, 'GUEST') AS plan_at_session
    FROM session_base sb
    LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_member') dm
        ON sb.member_id = dm.member_id
        AND sb.session_date >= dm.effective_from
        AND (sb.session_date < dm.effective_to OR dm.effective_to IS NULL)
),
tariff_lookup AS (
    SELECT
        mv.*,
        t.rate_per_kwh,
        t.free_idle_minutes,
        t.idle_fee_per_min
    FROM member_version mv
    LEFT JOIN IDENTIFIER(:catalog || '.silver.tariffs') t
        ON mv.connector_type = t.connector_type
        AND mv.time_band = t.time_band
),
fault_counts AS (
    SELECT session_id, COUNT(*) AS fault_count
    FROM IDENTIFIER(:catalog || '.silver.session_faults')
    GROUP BY session_id
),
billing AS (
    SELECT
        tl.*,
        ROUND(tl.energy_kwh * tl.rate_per_kwh, 2) AS energy_charge_calc,
        ROUND(ROUND(tl.energy_kwh * tl.rate_per_kwh, 2) *
            CASE tl.plan_at_session
                WHEN 'BASIC' THEN 0.00
                WHEN 'PRO'   THEN 0.10
                WHEN 'FLEET' THEN 0.15
                ELSE 0.00
            END, 2) AS plan_discount_calc,
        ROUND(GREATEST(tl.idle_minutes - tl.free_idle_minutes, 0) * tl.idle_fee_per_min, 2) AS idle_fee_calc
    FROM tariff_lookup tl
)
SELECT
    b.session_id,
    b.date_key,
    b.charger_id,
    b.station_id,
    b.member_id,
    b.status,
    b.time_band,
    b.energy_kwh,
    b.duration_minutes,
    b.idle_minutes,
    b.amount_paid,
    b.member_sk,
    b.plan_at_session,
    b.rate_per_kwh,
    CASE WHEN b.status = 'COMPLETED' THEN b.energy_charge_calc    ELSE 0.00 END AS energy_charge,
    CASE WHEN b.status = 'COMPLETED' THEN b.plan_discount_calc    ELSE 0.00 END AS plan_discount,
    CASE WHEN b.status = 'COMPLETED' THEN b.idle_fee_calc        ELSE 0.00 END AS idle_fee,
    CASE WHEN b.status = 'COMPLETED' THEN ROUND(b.energy_charge_calc - b.plan_discount_calc + b.idle_fee_calc, 2) ELSE 0.00 END AS billed_amount,
    CASE WHEN b.status = 'COMPLETED' THEN
        CASE WHEN ABS(b.amount_paid - ROUND(b.energy_charge_calc - b.plan_discount_calc + b.idle_fee_calc, 2)) <= 0.50
            THEN 'OK' ELSE 'MISMATCH' END
        ELSE NULL END AS billing_status,
    COALESCE(fc.fault_count, 0) AS fault_count
FROM billing b
LEFT JOIN fault_counts fc ON b.session_id = fc.session_id


In [0]:
CREATE MATERIALIZED VIEW IF NOT EXISTS IDENTIFIER(:catalog || '.gold.agg_daily_station') AS SELECT
    session_id,
    member_id,
    d.calendar_date AS session_date,
    status,
    member_sk,                                      -- version in force on session_date; NULL for guests
    plan_at_session,                                -- plan of that version; GUEST for guests
    c.connector_type,
    fs.time_band,
    fs.rate_per_kwh,                                   -- from silver.tariffs by connector_type & time_band
    energy_kwh,
    CASE WHEN status = 'COMPLETED'
         THEN ROUND(energy_kwh * fs.rate_per_kwh, 2)
         ELSE 0.00
    END AS energy_charge,                           -- ROUND(energy_kwh × rate_per_kwh)
    CASE WHEN status = 'COMPLETED'
         THEN ROUND(
                ROUND(energy_kwh * fs.rate_per_kwh, 2) *
                CASE plan_at_session
                    WHEN 'BASIC' THEN 0.00
                    WHEN 'PRO'   THEN 0.10
                    WHEN 'FLEET' THEN 0.15
                    ELSE 0.00
                END, 2)
         ELSE 0.00
    END AS plan_discount,                           -- ROUND(energy_charge × rate): BASIC 0%, PRO 10%, FLEET 15%, GUEST 0%
    CASE WHEN status = 'COMPLETED'
         THEN ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2)
         ELSE 0.00
    END AS idle_fee,                                -- max(idle_minutes - free_idle_minutes, 0) × idle_fee_per_min
    CASE WHEN status = 'COMPLETED'
         THEN ROUND(
              ROUND(energy_kwh * fs.rate_per_kwh, 2)
              - ROUND(ROUND(energy_kwh * fs.rate_per_kwh, 2) *
                      CASE plan_at_session
                          WHEN 'BASIC' THEN 0.00
                          WHEN 'PRO'   THEN 0.10
                          WHEN 'FLEET' THEN 0.15
                          ELSE 0.00
                      END, 2)
              + ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2),
              2)
         ELSE 0.00
    END AS billed_amount,                           -- energy_charge - plan_discount + idle_fee
    CASE WHEN status = 'COMPLETED'
         THEN CASE WHEN ABS(amount_paid -
                 ROUND(
                   ROUND(energy_kwh * fs.rate_per_kwh, 2)
                   - ROUND(ROUND(energy_kwh * fs.rate_per_kwh, 2) *
                           CASE plan_at_session
                               WHEN 'BASIC' THEN 0.00
                               WHEN 'PRO'   THEN 0.10
                               WHEN 'FLEET' THEN 0.15
                               ELSE 0.00
                           END, 2)
                   + ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2),
                   2)
                 ) <= 0.50 THEN 'OK' ELSE 'MISMATCH' END
         ELSE NULL
    END AS billing_status,                          -- OK if |amount_paid - billed_amount| ≤ 0.50, else MISMATCH; NULL for non-COMPLETED
    (SELECT COUNT(*) FROM IDENTIFIER(:catalog || '.silver.session_faults') sf
     WHERE sf.session_id = fs.session_id) AS fault_count  -- faults of the session in silver.session_faults
FROM IDENTIFIER(:catalog || '.gold.fact_sessions') fs
LEFT JOIN IDENTIFIER(:catalog || '.silver.chargers') c
    ON fs.charger_id = c.charger_id
LEFT JOIN IDENTIFIER(:catalog || '.silver.tariffs') t
    ON c.connector_type = t.connector_type
    AND fs.time_band = t.time_band
LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_date') d
    ON fs.date_key = d.date_key
ORDER BY session_date, session_id;

In [0]:
CREATE MATERIALIZED VIEW IF NOT EXISTS IDENTIFIER(:catalog || '.gold.agg_daily_station') AS 
SELECT
    s.session_id,
    s.member_id,
    s.session_date,
    s.status,
    m.member_sk,                                                          -- version in force on session_date; NULL for guests
    COALESCE(m.plan, 'GUEST')                  AS plan_at_session,          -- plan of that version; GUEST for guests
    t.rate_per_kwh,                                                       -- from silver.tariffs by connector_type & time_band
    CASE WHEN s.status = 'COMPLETED'
         THEN ROUND(s.energy_kwh * t.rate_per_kwh, 2)
         ELSE 0.00
    END                                       AS energy_charge,            -- ROUND(energy_kwh × rate_per_kwh)
    CASE WHEN s.status = 'COMPLETED'
         THEN ROUND(ROUND(s.energy_kwh * t.rate_per_kwh, 2) *
              CASE COALESCE(m.plan, 'GUEST')
                  WHEN 'BASIC' THEN 0.00
                  WHEN 'PRO'   THEN 0.10
                  WHEN 'FLEET' THEN 0.15
                  ELSE 0.00
              END, 2)
         ELSE 0.00
    END                                       AS plan_discount,           -- ROUND(energy_charge × rate): BASIC 0%, PRO 10%, FLEET 15%, GUEST 0%
    CASE WHEN s.status = 'COMPLETED'
         THEN ROUND(GREATEST(s.idle_minutes - t.free_idle_minutes, 0) * t.idle_fee_per_min, 2)
         ELSE 0.00
    END                                       AS idle_fee,                -- max(idle_minutes - free_idle_minutes, 0) × idle_fee_per_min
    CASE WHEN s.status = 'COMPLETED'
         THEN ROUND(
              ROUND(s.energy_kwh * t.rate_per_kwh, 2)
              - ROUND(ROUND(s.energy_kwh * t.rate_per_kwh, 2) *
                      CASE COALESCE(m.plan, 'GUEST')
                          WHEN 'BASIC' THEN 0.00
                          WHEN 'PRO'   THEN 0.10
                          WHEN 'FLEET' THEN 0.15
                          ELSE 0.00
                      END, 2)
              + ROUND(GREATEST(s.idle_minutes - t.free_idle_minutes, 0) * t.idle_fee_per_min, 2),
              2)
         ELSE 0.00
    END                                       AS billed_amount,           -- energy_charge - plan_discount + idle_fee
    CASE WHEN s.status = 'COMPLETED'
         THEN CASE WHEN ABS(s.amount_paid -
              ROUND(
                ROUND(s.energy_kwh * t.rate_per_kwh, 2)
                - ROUND(ROUND(s.energy_kwh * t.rate_per_kwh, 2) *
                        CASE COALESCE(m.plan, 'GUEST')
                            WHEN 'BASIC' THEN 0.00
                            WHEN 'PRO'   THEN 0.10
                            WHEN 'FLEET' THEN 0.15
                            ELSE 0.00
                        END, 2)
                + ROUND(GREATEST(s.idle_minutes - t.free_idle_minutes, 0) * t.idle_fee_per_min, 2),
                2)
              ) <= 0.50 THEN 'OK' ELSE 'MISMATCH' END
         ELSE NULL
    END                                       AS billing_status,           -- OK if |amount_paid - billed_amount| ≤ 0.50, else MISMATCH; NULL for non-COMPLETED
    (SELECT COUNT(*)
       FROM IDENTIFIER(:catalog || '.silver.session_faults') sf
      WHERE sf.session_id = s.session_id)       AS fault_count              -- faults of the session in silver.session_faults
FROM IDENTIFIER(:catalog || '.silver.sessions') s
LEFT JOIN IDENTIFIER(:catalog || '.silver.chargers') c
    ON s.charger_id = c.charger_id
LEFT JOIN IDENTIFIER(:catalog || '.silver.tariffs') t
    ON c.connector_type = t.connector_type
    AND s.time_band = t.time_band
LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_member') m
    ON s.member_id = m.member_id
    AND s.session_date >= m.effective_from
    AND (s.session_date < m.effective_to OR m.effective_to IS NULL)
ORDER BY s.session_date, s.session_id;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.agg_daily_station') AS
SELECT
    date_key,
    station_id,
    SUM(energy_kwh) AS energy_kwh,
    SUM(billed_amount) AS total_billed,
    COUNT(*) AS session_count
FROM IDENTIFIER(:catalog || '.gold.fact_sessions')
WHERE status = 'COMPLETED'
GROUP BY date_key, station_id